In [13]:
# Import necessary libraries
import pandas as pd
import mlflow
import mlflow.sklearn
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from solids.data_preprocessing import read_and_clean_data
from solids.feature_engineering import create_spreads_and_more

# Set the file path for the data
file_path = 'data/30_yr_stock_market_data.csv'

# Read and clean data
data = read_and_clean_data(file_path)

# Create spreads and more
X, y = create_spreads_and_more(data)

# Set MLflow tracking URI
mlflow.set_tracking_uri("http://localhost:5001")

# Start an MLflow experiment
mlflow.set_experiment("Stock Market Prediction")

# Train k-Nearest Neighbors model and log the experiment
with mlflow.start_run(run_name="kNN") as run:
    Number_trials = 3
    n_neighbors_tuning = (2, 6)
    weights_tuning = ['uniform', 'distance']
    score_train = []
    score_test = []
    
    for seed in range(Number_trials):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
        for n_neighbors in n_neighbors_tuning:
            for weights in weights_tuning:
                start_time = time.time()  # Time the training process
                knn = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights).fit(X_train, y_train)
                
                # Calculate Mean Squared Error as a performance metric
                training_mse = mean_squared_error(y_train, knn.predict(X_train))
                test_mse = mean_squared_error(y_test, knn.predict(X_test))
                
                run_time = time.time() - start_time
                
                score_train.append((n_neighbors, weights, training_mse))
                score_test.append((n_neighbors, weights, test_mse))

    train_score = np.array([x[2] for x in score_train])
    test_score = np.array([x[2] for x in score_test])
    best_index = np.argmin(test_score)
    
    best_knn = KNeighborsRegressor(n_neighbors=score_test[best_index][0], 
                                   weights=score_test[best_index][1]).fit(X_train, y_train)
    
    # Log the model and metrics to MLflow
    input_example = X_test.iloc[:5]  # Provide an example input
    mlflow.sklearn.log_model(best_knn, "kNN_model", input_example=input_example)
    mlflow.log_param("n_neighbors", score_test[best_index][0])
    mlflow.log_param("weights", score_test[best_index][1])
    mlflow.log_metric("train_score", train_score[best_index])
    mlflow.log_metric("test_score", test_score[best_index])
    mlflow.log_metric("run_time", run_time)

# Train Gradient Boosting Machine model and log the experiment
with mlflow.start_run(run_name="GBM") as run:
    Number_trials = 3
    n_estimators_tuning = (50, 100)
    learning_rate_tuning = (0.01, 0.1)
    score_train = []
    score_test = []
    
    for seed in range(Number_trials):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
        for n_estimators in n_estimators_tuning:
            for learning_rate in learning_rate_tuning:
                start_time = time.time()  # Time the training process
                gbm = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate).fit(X_train, y_train)
                
                # Calculate Mean Squared Error as a performance metric
                training_mse = mean_squared_error(y_train, gbm.predict(X_train))
                test_mse = mean_squared_error(y_test, gbm.predict(X_test))
                
                run_time = time.time() - start_time
                
                score_train.append((n_estimators, learning_rate, training_mse))
                score_test.append((n_estimators, learning_rate, test_mse))

    train_score = np.array([x[2] for x in score_train])
    test_score = np.array([x[2] for x in score_test])
    best_index = np.argmin(test_score)
    
    best_gbm = GradientBoostingRegressor(n_estimators=score_test[best_index][0], 
                                         learning_rate=score_test[best_index][1]).fit(X_train, y_train)
    
    # Log the model and metrics to MLflow
    input_example = X_test.iloc[:5]  # Provide an example input
    mlflow.sklearn.log_model(best_gbm, "GBM_model", input_example=input_example)
    mlflow.log_param("n_estimators", score_test[best_index][0])
    mlflow.log_param("learning_rate", score_test[best_index][1])
    mlflow.log_metric("train_score", train_score[best_index])
    mlflow.log_metric("test_score", test_score[best_index])
    mlflow.log_metric("run_time", run_time)

# Train Random Forest model and log the experiment
with mlflow.start_run(run_name="RF") as run:
    Number_trials = 3
    n_estimators_tuning = (50, 100)
    max_depth_tuning = (10, 20)
    score_train = []
    score_test = []
    
    for seed in range(Number_trials):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
        for n_estimators in n_estimators_tuning:
            for max_depth in max_depth_tuning:
                start_time = time.time()  # Time the training process
                rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth).fit(X_train, y_train)
                
                # Calculate Mean Squared Error as a performance metric
                training_mse = mean_squared_error(y_train, rf.predict(X_train))
                test_mse = mean_squared_error(y_test, rf.predict(X_test))
                
                run_time = time.time() - start_time
                
                score_train.append((n_estimators, max_depth, training_mse))
                score_test.append((n_estimators, max_depth, test_mse))

    train_score = np.array([x[2] for x in score_train])
    test_score = np.array([x[2] for x in score_test])
    best_index = np.argmin(test_score)
    
    best_rf = RandomForestRegressor(n_estimators=score_test[best_index][0], 
                                    max_depth=score_test[best_index][1]).fit(X_train, y_train)
    
    # Log the model and metrics to MLflow
    input_example = X_test.iloc[:5]  # Provide an example input
    mlflow.sklearn.log_model(best_rf, "RF_model", input_example=input_example)
    mlflow.log_param("n_estimators", score_test[best_index][0])
    mlflow.log_param("max_depth", score_test[best_index][1])
    mlflow.log_metric("train_score", train_score[best_index])
    mlflow.log_metric("test_score", test_score[best_index])
    mlflow.log_metric("run_time", run_time)

print("MLflow runs completed. Check the MLflow UI for details.")

/Users/samnmercado/Documents/AIM/Sem 7/7 DASCI 270 ML Ops/HW1 Stocks Data/solids/data_preprocessing.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Date'] = pd.to_datetime(data['Date'])
/Users/samnmercado/Documents/AIM/Sem 7/7 DASCI 270 ML Ops/HW1 Stocks Data/dagster-env/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See

🏃 View run kNN at: http://localhost:5001/#/experiments/1/runs/dbc3ef109fa3433eb7fe931eea248181
🧪 View experiment at: http://localhost:5001/#/experiments/1


/Users/samnmercado/Documents/AIM/Sem 7/7 DASCI 270 ML Ops/HW1 Stocks Data/dagster-env/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run GBM at: http://localhost:5001/#/experiments/1/runs/7135c6ee0268450a98937350d26d708a
🧪 View experiment at: http://localhost:5001/#/experiments/1


/Users/samnmercado/Documents/AIM/Sem 7/7 DASCI 270 ML Ops/HW1 Stocks Data/dagster-env/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run RF at: http://localhost:5001/#/experiments/1/runs/c22c3bb69da848c897da3c8f492eabf5
🧪 View experiment at: http://localhost:5001/#/experiments/1
MLflow runs completed. Check the MLflow UI for details.
